# Real-Time Customer Purchase Behavior Prediction Using Machine Learning-Driven Segmentation

**Final Year Project**

This notebook provides a complete end-to-end implementation for your final year project. It covers data loading, preprocessing, segmentation (K-Means), and predictive modeling (Random Forest) for predicting next purchase behavior.
Dataset: UCI Online Retail Dataset

## 1. Setup and Installation
Run the following cell to install necessary libraries (if not already present in Colab).

In [ ]:
!pip install lifetimes openpyxl
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import datetime as dt
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

import warnings
warnings.filterwarnings('ignore')

## 2. Data Loading
We will download the Online Retail dataset directly from the UCI Machine Learning Repository.

In [ ]:
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00352/Online%20Retail.xlsx"
print("Downloading and loading the dataset. This might take a minute...")
df = pd.read_excel(url)
print("Dataset loaded successfully!")
df.head()

## 3. Data Preprocessing & Cleaning
Handle missing values, remove cancelled orders, and clean the data.

In [ ]:
# Remove rows with missing CustomerID
df_clean = df.dropna(subset=['CustomerID'])

# Remove cancelled orders (InvoiceNo starts with 'C')
df_clean = df_clean[~df_clean['InvoiceNo'].astype(str).str.startswith('C')]

# Remove rows with negative or zero quantity and price
df_clean = df_clean[(df_clean['Quantity'] > 0) & (df_clean['UnitPrice'] > 0)]

# Calculate Total Price
df_clean['TotalAmount'] = df_clean['Quantity'] * df_clean['UnitPrice']

# Convert CustomerID to integer
df_clean['CustomerID'] = df_clean['CustomerID'].astype(int)

print(f"Original dataset shape: {df.shape}")
print(f"Cleaned dataset shape: {df_clean.shape}")

## 4. Feature Engineering: RFM Analysis
RFM (Recency, Frequency, Monetary) analysis is a marketing technique used to quantitatively rank and group customers based on the recency, frequency and monetary total of their recent transactions.

In [ ]:
# Define an analysis date (one day after the last transaction in the dataset)
analysis_date = df_clean['InvoiceDate'].max() + dt.timedelta(days=1)

rfm = df_clean.groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (analysis_date - x.max()).days, # Recency
    'InvoiceNo': 'nunique', # Frequency
    'TotalAmount': 'sum' # Monetary Component
})

rfm.rename(columns={'InvoiceDate': 'Recency',
                    'InvoiceNo': 'Frequency',
                    'TotalAmount': 'Monetary'}, inplace=True)
rfm.head()

## 5. Machine Learning-Driven Customer Segmentation
We use K-Means clustering to segment our customers based on their RFM behavior.

In [ ]:
# Scale the data for K-Means
scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm)

# Apply K-Means clustering
kmeans = KMeans(n_clusters=4, random_state=42)
kmeans.fit(rfm_scaled)

# Assign clusters back to the RFM dataframe
rfm['Cluster'] = kmeans.labels_

# Visualize the segments
plt.figure(figsize=(10, 6))
sns.scatterplot(data=rfm, x='Recency', y='Monetary', hue='Cluster', palette='viridis')
plt.title('Customer Segments: Recency vs Monetary')
plt.show()

print("Segment Averages:")
print(rfm.groupby('Cluster').mean())

## 6. Real-Time Purchase Prediction Dataset Construction
To predict future purchases, we split our data chronologically. 
We'll use data up to a certain point (e.g., first 9 months) to predict if a customer will purchase in the final 3 months.

In [ ]:
# Define the cut-off date for our features
cut_off_date = df_clean['InvoiceDate'].max() - pd.DateOffset(months=3)

# Data for Feature Engineering (Time Period 1: TP1)
tp1_data = df_clean[df_clean['InvoiceDate'] < cut_off_date]

# Data for Target Variable (Time Period 2: TP2)
tp2_data = df_clean[df_clean['InvoiceDate'] >= cut_off_date]

# Build Features from TP1
features = tp1_data.groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (cut_off_date - x.max()).days,
    'InvoiceNo': 'nunique',
    'TotalAmount': ['sum', 'mean']
})
features.columns = ['Recency', 'Frequency', 'Total_Spend', 'Avg_Order_Value']

# Add the Segment from earlier (simulating that segment history is a feature)
features = features.merge(rfm[['Cluster']], on='CustomerID', how='left')

# Define target: 1 if customer purchased in TP2, else 0
tp2_purchasers = tp2_data['CustomerID'].unique()
features['Target'] = features.index.isin(tp2_purchasers).astype(int)

features.fillna(0, inplace=True)
features.head()

## 7. Predictive Modeling (Random Forest)
Training a machine learning model to predict next purchase behavior.

In [ ]:
X = features.drop('Target', axis=1)
y = features['Target']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

y_pred = rf_model.predict(X_test)

print("Accuracy Score:", accuracy_score(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

## 8. Real-Time Inference Simulation
Simulating how a new customer feature vector would be scored in real-time.

In [ ]:
# Let's say we receive an event from a customer with the following features:
# Recency: 15 days, Frequency: 5 orders, Total_Spend: 450, Avg_Order_Value: 90, Cluster: 1
real_time_event = pd.DataFrame([{
    'Recency': 15,
    'Frequency': 5,
    'Total_Spend': 450,
    'Avg_Order_Value': 90,
    'Cluster': 1
}])

prediction_prob = rf_model.predict_proba(real_time_event)[0][1]
print(f"Probability of customer purchasing in the next time window: {prediction_prob:.2%}")

if prediction_prob > 0.5:
    print("Action: High intent. No discount required right now, customer is likely to purchase naturally.")
else:
    print("Action: Low intent. Send targeted promotional email based on cluster preferences.")